# INV-M365-CF — Authoritative Persona Post-H5 Department-Pack Coverage Parity Correction v1

**Purpose:** Prove the final post-H5 department-pack parity targets for the 7 affected packs that still block fresh `M1` replay.

**Lemma mapping:** `docs/ma/lemmas/L84_m365_authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1.md`

**Invariant support:** `invariants/lemmas/L84_m365_authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1.yaml`

**Expected results:** Deterministically derive the final affected-pack coverage, action, and pack-state targets; preserve the current `department.status` taxonomy; and overwrite `configs/generated/authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1_verification.json` with the proof-backed target matrix used for extraction.

This notebook is the Phase 5 source for the post-H5 department-pack parity correction.
It proves the source-branch pack contracts must match the already-truthful registry, while leaving the downstream certification taxonomy untouched in this phase.

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import yaml

repo = Path.cwd().resolve()
while not (repo / "registry" / "persona_registry_v2.yaml").exists():
    if repo.parent == repo:
        raise RuntimeError("repo root not found")
    repo = repo.parent

paths = {
    "registry": repo / "registry" / "persona_registry_v2.yaml",
    "verification_output": repo / "configs" / "generated" / "authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1_verification.json",
    "lemma": repo / "docs" / "ma" / "lemmas" / "L84_m365_authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1.md",
    "invariant": repo / "invariants" / "lemmas" / "L84_m365_authoritative_persona_post_h5_department_pack_coverage_parity_correction_v1.yaml",
}

registry = yaml.safe_load(paths["registry"].read_text(encoding="utf-8"))
existing_verification = {}
if paths["verification_output"].exists():
    existing_verification = json.loads(paths["verification_output"].read_text(encoding="utf-8"))

department_pack_paths = sorted((repo / "registry").glob("department_pack_*_v1.yaml"))
department_packs = {}
for pack_path in department_pack_paths:
    payload = yaml.safe_load(pack_path.read_text(encoding="utf-8"))
    department_packs[payload["department"]["id"]] = payload

affected_departments_expected = [
    "communication",
    "engineering",
    "hr",
    "marketing",
    "operations",
    "project-management",
    "studio-operations",
]
existing_mismatches = existing_verification.get("affected_departments") or []
assert len(existing_mismatches) == 7
assert [item["department_id"] for item in existing_mismatches] == affected_departments_expected
assert existing_verification.get("mismatch_total") == 20


Load the live registry and current department-pack authorities.
The registry is already final. The affected pack contracts are the stale layer that still records staged contract-only coverage for promoted personas.

In [2]:
registry_personas = registry["personas"]
full_docs_scope = [
    "docs/commercialization/m365-communication-department-pack-v1.md",
    "docs/commercialization/m365-engineering-department-pack-v1.md",
    "docs/commercialization/m365-hr-department-pack-v1.md",
    "docs/commercialization/m365-marketing-department-pack-v1.md",
    "docs/commercialization/m365-operations-department-pack-v1.md",
    "docs/commercialization/m365-project-management-department-pack-v1.md",
    "docs/commercialization/m365-studio-operations-department-pack-v1.md",
]
full_verifier_scope = [
    "scripts/ci/verify_communication_department_pack_v1.py",
    "scripts/ci/verify_engineering_department_pack_v1.py",
    "scripts/ci/verify_hr_department_pack_v1.py",
    "scripts/ci/verify_marketing_department_pack_v1.py",
    "scripts/ci/verify_operations_department_pack_v1.py",
    "scripts/ci/verify_project_management_department_pack_v1.py",
    "scripts/ci/verify_studio_operations_department_pack_v1.py",
]
full_test_scope = [
    "tests/test_communication_department_pack_v1.py",
    "tests/test_engineering_department_pack_v1.py",
    "tests/test_hr_department_pack_v1.py",
    "tests/test_marketing_department_pack_v1.py",
    "tests/test_operations_department_pack_v1.py",
    "tests/test_project_management_department_pack_v1.py",
    "tests/test_studio_operations_department_pack_v1.py",
]

target_department_summaries = {}
affected_departments = []
for department_id in affected_departments_expected:
    pack_payload = department_packs[department_id]
    mismatches = []
    target_persona_rebases = []
    persona_count = 0
    active_persona_count = 0
    planned_persona_count = 0
    registry_backed_persona_count = 0
    contract_only_persona_count = 0
    supported_action_count = 0

    for persona_id, declared in pack_payload["personas"].items():
        registry_persona = registry_personas[persona_id]
        persona_count += 1
        if registry_persona["status"] == "active":
            active_persona_count += 1
        else:
            planned_persona_count += 1
        if registry_persona["coverage_status"] == "registry-backed":
            registry_backed_persona_count += 1
        else:
            contract_only_persona_count += 1
        supported_action_count += len(registry_persona.get("allowed_actions") or [])

        declared_actions = declared.get("supported_actions") or []
        registry_actions = registry_persona.get("allowed_actions") or []
        if declared["coverage_status"] != registry_persona["coverage_status"] or declared_actions != registry_actions:
            mismatches.append(
                {
                    "persona_id": persona_id,
                    "declared_coverage_status": declared["coverage_status"],
                    "registry_coverage_status": registry_persona["coverage_status"],
                    "declared_supported_action_count": len(declared_actions),
                    "registry_allowed_action_count": len(registry_actions),
                }
            )
            target_persona_rebases.append(
                {
                    "persona_id": persona_id,
                    "display_name": registry_persona["display_name"],
                    "title": registry_persona["title"],
                    "status": registry_persona["status"],
                    "final_coverage_status": registry_persona["coverage_status"],
                    "final_supported_action_count": len(registry_actions),
                    "final_supported_actions": list(registry_actions),
                }
            )

    target_pack_state = "blocked"
    if planned_persona_count == 0:
        target_pack_state = "ready"

    target_summary = {
        "persona_count": persona_count,
        "active_persona_count": active_persona_count,
        "planned_persona_count": planned_persona_count,
        "registry_backed_persona_count": registry_backed_persona_count,
        "contract_only_persona_count": contract_only_persona_count,
        "supported_action_count": supported_action_count,
        "workflow_family_count": len(pack_payload.get("workflow_families") or []),
        "workload_family_count": len(pack_payload.get("workload_families") or []),
        "department_status": pack_payload["department"]["status"],
        "target_pack_state": target_pack_state,
    }
    target_department_summaries[department_id] = target_summary
    affected_departments.append(
        {
            "department_id": department_id,
            "pack_path": str((repo / "registry" / f"department_pack_{department_id.replace('-', '_')}_v1.yaml").relative_to(repo)),
            "mismatch_count": len(mismatches),
            "mismatches": mismatches,
            "target_summary": target_summary,
            "target_persona_rebases": target_persona_rebases,
        }
    )

assert sum(item["mismatch_count"] for item in affected_departments) == 20
assert target_department_summaries["communication"] == {
    "persona_count": 4,
    "active_persona_count": 4,
    "planned_persona_count": 0,
    "registry_backed_persona_count": 4,
    "contract_only_persona_count": 0,
    "supported_action_count": 49,
    "workflow_family_count": 3,
    "workload_family_count": 5,
    "department_status": "partial-activation",
    "target_pack_state": "ready",
}
assert target_department_summaries["engineering"]["supported_action_count"] == 65
assert target_department_summaries["hr"] == {
    "persona_count": 2,
    "active_persona_count": 2,
    "planned_persona_count": 0,
    "registry_backed_persona_count": 2,
    "contract_only_persona_count": 0,
    "supported_action_count": 10,
    "workflow_family_count": 4,
    "workload_family_count": 5,
    "department_status": "partial-activation",
    "target_pack_state": "ready",
}
assert target_department_summaries["marketing"] == {
    "persona_count": 8,
    "active_persona_count": 3,
    "planned_persona_count": 5,
    "registry_backed_persona_count": 3,
    "contract_only_persona_count": 5,
    "supported_action_count": 24,
    "workflow_family_count": 4,
    "workload_family_count": 7,
    "department_status": "partial-activation",
    "target_pack_state": "blocked",
}
assert target_department_summaries["operations"]["supported_action_count"] == 85
assert target_department_summaries["project-management"]["supported_action_count"] == 40
assert target_department_summaries["studio-operations"]["supported_action_count"] == 61
assert all(item["target_summary"]["department_status"] == "partial-activation" for item in affected_departments)
assert any(item["department_id"] == "marketing" and item["target_summary"]["target_pack_state"] == "blocked" for item in affected_departments)
assert all(item["target_summary"]["target_pack_state"] == "ready" for item in affected_departments if item["department_id"] != "marketing")


The parity proof is green only if all 20 mismatches collapse to the final registry-backed action surface, the default pack-state projection stays bounded, and the already-certified `department.status` taxonomy stays unchanged.

In [3]:
verification = {
    "lemma_id": "L84",
    "fresh_m1_blocked_merge_commit": "d194b94",
    "fresh_m1_blocked_merge_branch": "development",
    "fresh_m1_blocked_merge_pushed": False,
    "fresh_m1_blocked_merge_safety_branch": "codex/m1-fresh-replay-blocked-development-d194b94",
    "previous_blocked_merge_safety_branch": "codex/m1-replay-blocked-development-a895678",
    "source_branch": "codex/m365-authoritative-persona-post-h5-parity-correction",
    "source_branch_head": "1cfc6c1",
    "origin_development_head": existing_verification.get("origin_development_head", "4c997ea"),
    "predecessor_blocker_proof": True,
    "blocked_by_m1_department_pack_validation": True,
    "affected_department_count": len(affected_departments),
    "mismatch_total": sum(item["mismatch_count"] for item in affected_departments),
    "affected_departments": affected_departments,
    "target_department_summaries": target_department_summaries,
    "runtime_change_required": False,
    "preserved_department_status_taxonomy": {
        department_id: summary["department_status"]
        for department_id, summary in target_department_summaries.items()
    },
    "required_scope": {
        "department_pack_authorities": [item["pack_path"] for item in affected_departments],
        "commercialization_contracts": full_docs_scope,
        "verifiers": full_verifier_scope,
        "tests": full_test_scope,
    },
}

normalized = json.dumps(verification, indent=2, sort_keys=True)
verification["verification_hash"] = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
normalized_with_hash = json.dumps(verification, indent=2, sort_keys=True)
assert verification["verification_hash"] == hashlib.sha256(json.dumps({k: v for k, v in verification.items() if k != "verification_hash"}, indent=2, sort_keys=True).encode("utf-8")).hexdigest()

paths["verification_output"].parent.mkdir(parents=True, exist_ok=True)
paths["verification_output"].write_text(normalized_with_hash + "\n", encoding="utf-8")
print(json.dumps({
    "lemma_id": verification["lemma_id"],
    "mismatch_total": verification["mismatch_total"],
    "affected_department_count": verification["affected_department_count"],
    "verification_hash": verification["verification_hash"],
}, indent=2, sort_keys=True))


{
  "affected_department_count": 7,
  "lemma_id": "L84",
  "mismatch_total": 20,
  "verification_hash": "3640dd0fc2da1aa007347cd0bfa4569809370f4a926b6ffe1c078791cfc79652"
}
